# 03 — Supporting analysis: K-means and demographic error audit
Supporting analyses only; neither creates a third research question. Cluster the four signed standardised discordance dimensions, without the readmission label. Fit clusters on training OOF residuals and transfer the fixed centres to held-out patients. Demographic audit uses frozen B and C predictions from the primary family. Do not infer treatment quality or discrimination from unadjusted differences.

In [ ]:
# Shared input setup (run from the project or statistical_analysis folder).
if (!requireNamespace("jsonlite", quietly=TRUE)) stop("Install jsonlite in R first; see README.")
root <- normalizePath(getwd(), winslash="/", mustWork=TRUE)
while (!file.exists(file.path(root,"statistical_analysis/private_exports/manifest.json"))) {
  parent <- dirname(root)
  if (identical(parent,root)) stop("Run 00_export_for_R.ipynb first.")
  root <- parent
}
input <- file.path(root,"statistical_analysis/private_exports")
output <- file.path(root,"statistical_analysis/outputs")
dir.create(output,recursive=TRUE,showWarnings=FALSE)
meta <- jsonlite::fromJSON(file.path(input,"manifest.json"))
read_split <- function(name) {
  d <- read.csv(file.path(input,paste0(name,".csv")), na.strings=c(""), check.names=FALSE,
                stringsAsFactors=FALSE)
  stopifnot(nrow(d)==meta$rows[[name]], all(d[[meta$target]] %in% c(0,1)),
            !anyNA(d$patient_cluster), all(is.finite(as.matrix(d[meta$discordance]))))
  d
}
train <- read_split("train")
test <- read_split("test")
y <- test[[meta$target]]
stopifnot(length(unique(y))==2)
cat("Training encounters:", nrow(train), "| Test encounters:",nrow(test),"\n")

## K-means clustering
Prespecify k=3 as a simple descriptive grouping, not an empirically optimal number. Inspect training centroids and optionally report training-only silhouette on a sample. Negative/positive coordinates mean below/above statistical expectation; clusters may have mixed patterns.

In [ ]:
x <- as.matrix(train[meta$discordance]); xt <- as.matrix(test[meta$discordance])
set.seed(42)
km <- kmeans(x,centers=3,nstart=20,iter.max=100)
if (km$ifault!=0) warning("K-means reported a convergence issue; inspect before interpreting.")
centres <- as.data.frame(km$centers); centres$cluster <- seq_len(nrow(centres)); centres$training_n <- km$size
print(centres); write.csv(centres,file.path(output,"cluster_centres.csv"),row.names=FALSE)
distances <- sapply(seq_len(nrow(km$centers)),function(k) rowSums(sweep(xt,2,km$centers[k,],"-")^2))
assignment <- max.col(-distances,ties.method="first")
cluster_summary <- do.call(rbind,lapply(seq_len(nrow(km$centers)),function(k) {
  i <- assignment==k
  data.frame(cluster=k,encounters=sum(i),patients=length(unique(test$patient_cluster[i])),
             readmission_rate=if (any(i)) mean(y[i]) else NA_real_)
}))
print(cluster_summary); write.csv(cluster_summary,file.path(output,"cluster_test_summary.csv"),row.names=FALSE)
plot_centres <- function() {
  old <- par(mar=c(8,4,3,1)); on.exit(par(old))
  matplot(t(km$centers),type="b",pch=1:3,lty=1:3,xaxt="n",xlab="",ylab="Mean signed residual / SD",main="Training cluster profiles")
  axis(1,at=seq_along(meta$discordance),labels=sub("discordance_","",meta$discordance),las=2,cex.axis=.8)
  abline(h=0,lty=2); legend("topright",legend=paste("Cluster",1:3),col=1:3,pch=1:3,lty=1:3,bty="n")
}
plot_centres(); png(file.path(output,"cluster_profiles.png"),width=1300,height=900,res=140); plot_centres(); dev.off()
if (requireNamespace("cluster",quietly=TRUE)) {
  set.seed(42); i <- sample.int(nrow(x),min(2000,nrow(x)))
  if (length(unique(km$cluster[i]))>1) print(mean(cluster::silhouette(km$cluster[i],dist(x[i,]))[,3]))
}

## Demographic subgroup error analysis
Compare recall, false-negative rate and false-positive rate for age, recorded gender and race separately. Suppress groups with fewer than 20 distinct patients, positive encounters or negative encounters; this is a pragmatic reporting rule, not proof of statistical reliability or anonymisation. Missing categories remain visible when sufficiently represented. Differences are descriptive and not causal. Do not change thresholds after the audit.

In [ ]:
m <- meta$models; rows <- list()
for (label in c("B","C")) {
  row <- m[m$model==meta$primary_family & m$feature_set==label,]
  stopifnot(nrow(row)==1)
  pred <- test[[row$prediction]]
  for (v in c("age","gender","race")) {
    grp <- as.character(test[[v]]); grp[is.na(grp)] <- "Missing"
    for (g in sort(unique(grp))) {
      i <- grp==g; pos <- sum(y[i]==1); neg <- sum(y[i]==0); patients <- length(unique(test$patient_cluster[i]))
      keep <- min(pos,neg,patients)>=20
      rows[[length(rows)+1L]] <- data.frame(feature_set=label,attribute=v,group=g,
        status=if (keep) "Reported" else "Suppressed",encounters=if (keep) sum(i) else NA_integer_,
        recall=if (keep) sum(pred[i]==1 & y[i]==1)/pos else NA_real_,
        FNR=if (keep) sum(pred[i]==0 & y[i]==1)/pos else NA_real_,
        FPR=if (keep) sum(pred[i]==1 & y[i]==0)/neg else NA_real_)
    }
  }
}
audit <- do.call(rbind,rows); print(audit)
write.csv(audit,file.path(output,"demographic_error_audit_R.csv"),row.names=FALSE)
gaps <- do.call(rbind,lapply(c("B","C"),function(label) do.call(rbind,lapply(c("age","gender","race"),function(v) {
  f <- audit$FNR[audit$feature_set==label & audit$attribute==v & audit$status=="Reported"]
  data.frame(feature_set=label,attribute=v,reported_groups=length(f),FNR_gap=if (length(f)>=2) diff(range(f)) else NA_real_)
}))))
print(gaps); write.csv(gaps,file.path(output,"demographic_FNR_gaps.csv"),row.names=FALSE)

## Discussion prompts
Describe cluster patterns and their observed readmission rates without assigning clinical adequacy labels. Compare subgroup error gaps across B and C, acknowledge uncertainty and small-group suppression. The dataset records historical US encounters, not current UK care. No measured financial or environmental costs are available. Report training time and workload proxies separately from actual sustainability benefits.